# Robot1 to Robot2 Transform Investigation

This notebook investigates the calibration data used to transform 3D points from Robot1 coordinates into Robot2 coordinates.

It uses the saved calibration points in this folder and the saved `calib_tool_tf-robot1_to_robot2.json` transform matrix. The core convention is:

```python
p_robot2 = R_robot1_to_robot2 @ p_robot1 + t_robot1_to_robot2
```

The plots show:

- Robot1 calibration points in Robot1 coordinates.
- Robot2 measured calibration points in Robot2 coordinates.
- Robot1 points transformed into Robot2 coordinates.
- Residual/error vectors between transformed Robot1 points and measured Robot2 points.
- Robot1 frame axes expressed inside Robot2 frame.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 - registers 3D projection

plt.rcParams["figure.figsize"] = (9, 7)
plt.rcParams["axes.grid"] = True

## Paths

The notebook can be run either from the repo root or from this calibration folder. The helper below finds the repo root by walking upward until it sees `denso_robot_bringup`.

In [ ]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "denso_robot_bringup").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")

REPO_ROOT = find_repo_root(Path.cwd())
POINTS_DIR = REPO_ROOT / "denso_robot_bringup" / "calibration_points09_04"
TRANSFORM_JSON = REPO_ROOT / "calib_tool_tf-robot1_to_robot2.json"

print("REPO_ROOT:", REPO_ROOT)
print("POINTS_DIR:", POINTS_DIR)
print("TRANSFORM_JSON:", TRANSFORM_JSON)

## Load Calibration Data

Each point has two JSON files:

- `calib_tool_tf-P{i}.json`: point measured from Robot1/world perspective.
- `calib_tool_tf-P{i}-R2.json`: corresponding point measured from Robot2/world perspective.

In [ ]:
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as infile:
        return json.load(infile)


def load_position(path: Path) -> np.ndarray:
    data = load_json(path)
    p = data["position"]
    return np.array([p["x"], p["y"], p["z"]], dtype=float)


def point_indices(points_dir: Path) -> list[int]:
    indices = []
    for path in points_dir.glob("calib_tool_tf-P*.json"):
        stem = path.stem
        if stem.endswith("-R2"):
            continue
        suffix = stem.replace("calib_tool_tf-P", "")
        if suffix.isdigit():
            indices.append(int(suffix))
    return sorted(indices)

indices = point_indices(POINTS_DIR)
labels = [f"P{i}" for i in indices]
points_r1 = np.vstack([load_position(POINTS_DIR / f"calib_tool_tf-P{i}.json") for i in indices])
points_r2_measured = np.vstack([load_position(POINTS_DIR / f"calib_tool_tf-P{i}-R2.json") for i in indices])

print(f"Loaded {len(indices)} point pairs")
print("Robot1 first point:", labels[0], points_r1[0])
print("Robot2 first point:", labels[0], points_r2_measured[0])

## Load and Apply the Saved Robot1 -> Robot2 Transform

In [ ]:
def load_transform_matrix(path: Path) -> np.ndarray:
    data = load_json(path)
    matrix = np.array(data["transform_robot1_to_robot2"]["matrix_4x4"], dtype=float)
    if matrix.shape != (4, 4):
        raise ValueError(f"Expected 4x4 matrix, got {matrix.shape}")
    return matrix


def transform_points(matrix_4x4: np.ndarray, points_xyz: np.ndarray) -> np.ndarray:
    rotation = matrix_4x4[:3, :3]
    translation = matrix_4x4[:3, 3]
    return (rotation @ points_xyz.T).T + translation

T_saved = load_transform_matrix(TRANSFORM_JSON)
R_saved = T_saved[:3, :3]
t_saved = T_saved[:3, 3]
points_r1_in_r2_saved = transform_points(T_saved, points_r1)

print("Saved T_robot1_to_robot2:")
np.set_printoptions(precision=8, suppress=True)
print(T_saved)
print("det(R):", np.linalg.det(R_saved))
print("translation [m]:", t_saved)

## Recompute the Transform From the Point Pairs

This recomputes a rigid transform using the Kabsch/SVD method. It is useful to check whether the saved matrix still matches the calibration point folder.

In [ ]:
def kabsch_transform(source_points: np.ndarray, target_points: np.ndarray) -> np.ndarray:
    source_centroid = source_points.mean(axis=0)
    target_centroid = target_points.mean(axis=0)

    source_centered = source_points - source_centroid
    target_centered = target_points - target_centroid

    covariance = source_centered.T @ target_centered
    U, singular_values, Vt = np.linalg.svd(covariance)
    rotation = Vt.T @ U.T

    if np.linalg.det(rotation) < 0.0:
        Vt[-1, :] *= -1.0
        rotation = Vt.T @ U.T

    translation = target_centroid - rotation @ source_centroid

    transform = np.eye(4)
    transform[:3, :3] = rotation
    transform[:3, 3] = translation
    return transform, singular_values

T_fit, singular_values = kabsch_transform(points_r1, points_r2_measured)
points_r1_in_r2_fit = transform_points(T_fit, points_r1)

print("Recomputed T_robot1_to_robot2:")
print(T_fit)
print("singular values:", singular_values)
print("matrix difference T_fit - T_saved:")
print(T_fit - T_saved)

## Error Metrics

Residuals are computed in Robot2 coordinates:

```python
residual = measured_robot2_point - transformed_robot1_point
```

In [ ]:
def summarize_errors(name: str, transformed_points: np.ndarray) -> dict:
    residuals = points_r2_measured - transformed_points
    errors = np.linalg.norm(residuals, axis=1)
    summary = {
        "name": name,
        "rmse_m": float(np.sqrt(np.mean(errors ** 2))),
        "mean_m": float(np.mean(errors)),
        "median_m": float(np.median(errors)),
        "max_m": float(np.max(errors)),
        "max_label": labels[int(np.argmax(errors))],
        "residuals": residuals,
        "errors": errors,
    }
    return summary

saved_summary = summarize_errors("saved transform", points_r1_in_r2_saved)
fit_summary = summarize_errors("recomputed Kabsch fit", points_r1_in_r2_fit)

for summary in [saved_summary, fit_summary]:
    print(summary["name"])
    print(f"  RMSE:   {summary['rmse_m'] * 1000:.3f} mm")
    print(f"  mean:   {summary['mean_m'] * 1000:.3f} mm")
    print(f"  median: {summary['median_m'] * 1000:.3f} mm")
    print(f"  max:    {summary['max_m'] * 1000:.3f} mm at {summary['max_label']}")

print()
print("Worst points using saved transform:")
for idx in np.argsort(saved_summary["errors"])[-8:][::-1]:
    r = saved_summary["residuals"][idx]
    print(
        f"{labels[idx]:>3}: error={saved_summary['errors'][idx] * 1000:7.3f} mm "
        f"residual_xyz_mm=({r[0]*1000:7.3f}, {r[1]*1000:7.3f}, {r[2]*1000:7.3f})"
    )

## Plot Helpers

In [ ]:
def set_axes_equal(ax):
    limits = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
    centers = limits.mean(axis=1)
    radius = 0.5 * np.max(limits[:, 1] - limits[:, 0])
    ax.set_xlim3d([centers[0] - radius, centers[0] + radius])
    ax.set_ylim3d([centers[1] - radius, centers[1] + radius])
    ax.set_zlim3d([centers[2] - radius, centers[2] + radius])


def label_points(ax, points: np.ndarray, labels: list[str], every: int = 1):
    for idx, label in enumerate(labels):
        if idx % every != 0:
            continue
        x, y, z = points[idx]
        ax.text(x, y, z, label, fontsize=8)


def scatter3(ax, points: np.ndarray, label: str, color: str, marker: str = "o"):
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], label=label, color=color, marker=marker, s=38)


def setup_3d_axis(ax, title: str, units: str = "m"):
    ax.set_title(title)
    ax.set_xlabel(f"X [{units}]")
    ax.set_ylabel(f"Y [{units}]")
    ax.set_zlabel(f"Z [{units}]")
    ax.legend()
    set_axes_equal(ax)

## 3D Plot: Raw Point Clouds in Their Own Robot Frames

This does **not** compare them in the same coordinate frame. It is a quick shape sanity check.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
scatter3(ax, points_r1, "Robot1 measured points", "tab:blue", "o")
scatter3(ax, points_r2_measured, "Robot2 measured points", "tab:orange", "^")
label_points(ax, points_r1, labels, every=3)
label_points(ax, points_r2_measured, labels, every=3)
setup_3d_axis(ax, "Raw calibration point clouds in their native frames")
plt.show()

## 3D Plot: Robot1 Points Transformed Into Robot2 Frame

Blue points are Robot1 points after applying the saved matrix. Orange points are the measured Robot2 points. Green arrows show the residual from transformed point to measured point.

In [ ]:
residuals_saved = saved_summary["residuals"]
errors_saved = saved_summary["errors"]

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
scatter3(ax, points_r1_in_r2_saved, "Robot1 transformed into Robot2", "tab:blue", "o")
scatter3(ax, points_r2_measured, "Robot2 measured", "tab:orange", "^")

ax.quiver(
    points_r1_in_r2_saved[:, 0], points_r1_in_r2_saved[:, 1], points_r1_in_r2_saved[:, 2],
    residuals_saved[:, 0], residuals_saved[:, 1], residuals_saved[:, 2],
    length=1.0,
    normalize=False,
    color="tab:green",
    linewidth=1.0,
)
label_points(ax, points_r2_measured, labels, every=2)
setup_3d_axis(ax, "Saved Robot1 -> Robot2 transform residuals")
plt.show()

## 3D Plot: Error Magnitude as Height

This plot maps each transformed point onto the XY plane of Robot2 and uses the Z axis as error magnitude in millimeters. It makes spatial error patterns easier to spot.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
error_mm = errors_saved * 1000.0
p = ax.scatter(
    points_r2_measured[:, 0],
    points_r2_measured[:, 1],
    error_mm,
    c=error_mm,
    cmap="viridis",
    s=55,
)
for idx, label in enumerate(labels):
    ax.text(points_r2_measured[idx, 0], points_r2_measured[idx, 1], error_mm[idx], label, fontsize=8)
ax.set_title("Residual magnitude by Robot2 XY location")
ax.set_xlabel("Robot2 X [m]")
ax.set_ylabel("Robot2 Y [m]")
ax.set_zlabel("error [mm]")
fig.colorbar(p, ax=ax, shrink=0.65, label="error [mm]")
plt.show()

## 3D Plot: Robot1 Frame Axes Expressed in Robot2 Frame

The origin of Robot1 in Robot2 coordinates is the translation vector `t`. The arrows are the columns of `R`, scaled for visibility.

In [ ]:
def draw_frame(ax, origin: np.ndarray, rotation: np.ndarray, name: str, scale: float = 0.08):
    colors = ["red", "green", "blue"]
    axis_names = ["x", "y", "z"]
    for i in range(3):
        direction = rotation[:, i] * scale
        ax.quiver(
            origin[0], origin[1], origin[2],
            direction[0], direction[1], direction[2],
            color=colors[i], linewidth=2,
        )
        tip = origin + direction
        ax.text(tip[0], tip[1], tip[2], f"{name}_{axis_names[i]}", color=colors[i])
    ax.scatter([origin[0]], [origin[1]], [origin[2]], color="black", s=45)
    ax.text(origin[0], origin[1], origin[2], name, color="black")

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
scatter3(ax, points_r2_measured, "Robot2 measured points", "tab:orange", "^")
draw_frame(ax, np.zeros(3), np.eye(3), "robot2", scale=0.08)
draw_frame(ax, t_saved, R_saved, "robot1_in_robot2", scale=0.08)
setup_3d_axis(ax, "Robot1 frame expressed in Robot2 coordinates")
plt.show()

## Inspect Any Robot1 Point

Change `point_robot1` below to test how an arbitrary Robot1 point is transformed into Robot2 coordinates.

In [ ]:
point_robot1 = np.array([0.40, 0.00, 0.01], dtype=float)
point_robot2 = transform_points(T_saved, point_robot1.reshape(1, 3))[0]

print("point_robot1 [m]:", point_robot1)
print("point_robot2 [m]:", point_robot2)
print("formula: p2 = R @ p1 + t")
print("R @ p1:", R_saved @ point_robot1)
print("t:", t_saved)

## Optional: Compare Against Saved Transformed Output JSON

The repo already contains `robot1_points_in_robot2_frame.json`. This cell checks whether that file matches the current transform and point files.

In [ ]:
transformed_json = POINTS_DIR / "robot1_points_in_robot2_frame.json"
if transformed_json.exists():
    data = load_json(transformed_json)
    saved_output_points = []
    saved_output_labels = []
    for item in data["points"]:
        p = item["position_robot2_transformed"]
        saved_output_points.append([p["x"], p["y"], p["z"]])
        saved_output_labels.append(item["label"])
    saved_output_points = np.array(saved_output_points, dtype=float)

    if saved_output_labels == labels:
        delta = saved_output_points - points_r1_in_r2_saved
        print("Max difference vs saved output JSON [mm]:", np.max(np.linalg.norm(delta, axis=1)) * 1000.0)
    else:
        print("Label order differs; inspect manually.")
else:
    print("No saved transformed output JSON found.")